# Feature Selection using MRMR

In [1]:
import core.constants as c
from core.utils import save_df_as_table_image, extract_subject_id
import os

import warnings

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from feature_engine.selection import SmartCorrelatedSelection
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedGroupKFold

from feature_engine.selection import MRMR

import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
data = pd.read_csv(c.RICKD_FEATURE_SELECTION_PART_2_DATA_FILE_3, index_col=0)
display(data.head())

data.info(verbose=True)

,speed_output,step_width_left,step_width_right,stride_rate_left,stride_rate_right,stride_length_left,stride_length_right,swing_time_left,swing_time_right,stance_time_left,...,pelvic_drop_peak_vel_left,pelvic_drop_peak_vel_right,vertical_oscillation_left,vertical_oscillation_right,age,height,weight,gender,dominantleg,is_injured
id,,,,,,,,,,,,,,,,,,,,,
100001_20110531T161051,2.489233,0.123419,0.123419,78.947368,78.947368,1.891817,1.891817,0.4350,0.420,0.320,...,-82.090017,-57.417845,96.433793,92.586134,47.0,172.0,61.9,female,left,True
100002_20110601T140505,2.722687,0.032922,0.032922,81.632653,81.632653,2.001175,2.001175,0.4450,0.420,0.290,...,-57.724685,-60.462411,86.521432,94.945518,37.0,173.4,70.6,male,left,True
100003_20110601T095930,2.949904,0.097273,0.097273,78.947368,78.947368,2.241927,2.241927,0.4475,0.445,0.315,...,-82.171073,-95.302664,82.680986,76.611379,51.0,186.0,86.5,male,right,True
100004_20110203T120721,2.688014,0.011401,0.011401,82.191781,82.191781,1.962250,1.962250,0.4400,0.460,0.290,...,-63.319119,-49.646132,92.593339,83.273183,35.0,175.6,59.0,male,left,True
100004_20140929T102035,2.928598,0.029021,0.029021,83.333333,82.758621,2.108590,2.123233,0.4200,0.430,0.300,...,-32.717949,-52.004473,87.481400,75.091789,39.0,175.0,61.0,male,left,False


<class 'pandas.core.frame.DataFrame'>
Index: 1813 entries, 100001_20110531T161051 to 201225_20140515T133244
Data columns (total 87 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   speed_output                    1813 non-null   float64
 1   step_width_left                 1813 non-null   float64
 2   step_width_right                1813 non-null   float64
 3   stride_rate_left                1813 non-null   float64
 4   stride_rate_right               1813 non-null   float64
 5   stride_length_left              1813 non-null   float64
 6   stride_length_right             1813 non-null   float64
 7   swing_time_left                 1813 non-null   float64
 8   swing_time_right                1813 non-null   float64
 9   stance_time_left                1813 non-null   float64
 10  stance_time_right               1813 non-null   float64
 11  pelvis_peak_drop_angle_left     1813 non-null   float64
 12  

In [3]:
from core.processing import preprocess_features
from core.model_selection import split_train_test_set
from core.utils import extract_subject_id

y_label = "is_injured"
y = data[y_label]
X = data.drop(columns=[y_label])

X_train, X_test, y_train, y_test = split_train_test_set(
    X, y, test_size=0.2, random_state=RANDOM_STATE, groups=extract_subject_id(X.index.to_series())
)

a = extract_subject_id(X_train.index.to_series())
b = extract_subject_id(X_test.index.to_series())

if np.intersect1d(a, b).size > 0:
    raise ValueError("Error: Some subjects are in both train and test sets")

print(f"Train vs Test split: {X_train.shape[0] / X.shape[0] * 100:.2f}% / {X_test.shape[0] / X.shape[0] * 100:.2f}%")
print(f"Train Shape: {X_train.shape} / Test Shape: {X_test.shape}")
print(f"Injured class distribution Train: {y_train.value_counts().iloc[0] / y_train.shape[0] * 100:.2f}%")
print(f"Injured class distribution Test: {y_test.value_counts().iloc[0] / y_test.shape[0] * 100:.2f}%")

Train vs Test split: 79.76% / 20.24%
Train Shape: (1446, 86) / Test Shape: (367, 86)
Injured class distribution Train: 64.11%
Injured class distribution Test: 61.04%


In [4]:
feature_categorical_columns = ["gender", "dominantleg"]

X_train_prep = preprocess_features(X_train, y_train, feature_categorical_columns=feature_categorical_columns, cat_drop=None)
X_train_prep = X_train_prep.drop(columns=["dominantleg_ambidextrous"])
display(X_train_prep)

X_test_prep = preprocess_features(X_test, y_test, feature_categorical_columns=feature_categorical_columns, cat_drop=None)
display(X_test_prep)

,speed_output,step_width_left,step_width_right,stride_rate_left,stride_rate_right,stride_length_left,stride_length_right,swing_time_left,swing_time_right,stance_time_left,...,pelvic_drop_peak_vel_right,vertical_oscillation_left,vertical_oscillation_right,age,height,weight,gender_female,gender_male,dominantleg_left,dominantleg_right
id,,,,,,,,,,,,,,,,,,,,,
100001_20110531T161051,-0.555838,2.003936,2.003936,-0.849221,-0.852246,-0.288871,-0.288443,0.159275,-0.181066,0.771886,...,0.796873,0.600301,0.297894,0.716796,-0.040618,-0.634605,1.0,0.0,1.0,0.0
100002_20110601T140505,-0.058240,-0.796692,-0.796692,-0.319711,-0.321852,0.062394,0.063329,0.412273,-0.181066,-0.143176,...,0.704902,-0.003297,0.438660,-0.099721,0.113783,0.029859,0.0,1.0,1.0,0.0
100003_20110601T095930,0.426066,1.194771,1.194771,-0.849221,-0.852246,0.835709,0.837762,0.475523,0.452271,0.619376,...,-0.347560,-0.237156,-0.655194,1.043402,1.503392,1.244226,0.0,1.0,0.0,1.0
100004_20110203T120721,-0.132145,-1.462716,-1.462716,-0.209457,-0.211413,-0.062637,-0.061881,0.285774,0.832273,-0.143176,...,1.031643,0.366442,-0.257736,-0.263025,0.356413,-0.856093,0.0,1.0,1.0,0.0
100004_20140929T102035,0.380651,-0.917416,-0.917416,0.015644,-0.099452,0.407420,0.455957,-0.220221,0.072269,0.161845,...,0.960402,0.055158,-0.745856,0.063582,0.290241,-0.703343,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201099_20150414T092041,-0.501092,-1.070573,-1.070573,-0.849221,-0.852246,-0.226171,-0.225652,-0.030473,0.072269,1.229418,...,-0.603241,0.020215,-0.783300,1.533313,-0.757480,0.297173,1.0,0.0,0.0,1.0
201100_20150409T155915,0.087294,1.551794,1.551794,-1.345003,-1.348858,0.671832,0.673648,0.791770,0.958940,1.076907,...,-1.114684,-0.719890,-0.052653,1.125054,-0.261191,0.747787,0.0,1.0,0.0,1.0
201101_20150413T143152,0.167513,1.000229,1.000229,0.982258,0.982295,-0.187268,-0.186693,-0.536468,-0.434401,-0.600708,...,0.115490,-0.639950,-1.208110,-1.406148,-1.143483,-0.359654,0.0,1.0,0.0,1.0


,speed_output,step_width_left,step_width_right,stride_rate_left,stride_rate_right,stride_length_left,stride_length_right,swing_time_left,swing_time_right,stance_time_left,...,pelvic_drop_peak_vel_right,vertical_oscillation_left,vertical_oscillation_right,age,height,weight,gender_female,gender_male,dominantleg_left,dominantleg_right
id,,,,,,,,,,,,,,,,,,,,,
100005_20110208T120837,-0.195737,-0.312625,-0.312625,1.493685,1.497271,-0.733344,-0.732986,-1.662360,-1.455947,-0.282590,...,0.690556,-0.857947,-1.030956,-0.388523,-1.364557,-0.964282,0.0,1.0,0.0,1.0
100009_20110302T140353,-0.625112,0.528068,0.528068,-0.080904,-0.082409,-0.651346,-0.650957,-0.067074,0.013931,0.168543,...,-0.531242,-0.752506,-0.553361,-0.474940,-1.333511,-1.590499,1.0,0.0,1.0,0.0
100015_20110329T121446,-0.637803,1.058469,1.058469,-1.386752,-1.480659,-0.159325,-0.121262,-0.133544,0.461285,2.273833,...,-1.849887,-0.779462,-0.041533,2.031151,0.839708,1.401424,0.0,1.0,0.0,1.0
100024_20110419T164319,-1.527490,0.889744,0.889744,-0.133780,-0.135456,-1.621618,-1.621597,-0.798247,-0.689054,0.920432,...,-0.363579,-1.304147,-0.718668,1.339816,0.042861,0.288151,1.0,0.0,0.0,1.0
100030_20110505T115803,-1.065275,1.708453,1.708453,-0.835090,-0.839033,-0.875314,-0.875010,-0.067074,-0.177792,1.145999,...,0.628543,-1.374771,-1.745295,0.994148,0.839708,1.109190,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201090_20150402T171016,0.449709,-1.029740,-1.029740,2.232669,2.238644,-0.345517,-0.345012,-1.130598,-1.711578,-1.635990,...,-0.399062,-0.768618,-1.521076,0.734897,-1.964779,-1.312180,1.0,0.0,0.0,1.0
201090_20150402T171510,1.277495,-0.849845,-0.849845,3.510630,3.360280,-0.001813,0.050793,-2.725884,-2.478470,-0.658535,...,-0.368810,-1.162345,-2.124031,0.734897,-1.964779,-1.312180,1.0,0.0,0.0,1.0
201092_20141208T180913,0.248739,0.204351,0.204351,-1.297737,-1.303176,0.855318,0.856279,2.591736,2.506332,-1.034479,...,-0.875103,2.748701,2.392628,0.734897,0.725873,0.148991,0.0,1.0,1.0,0.0


In [5]:
print("Train columns == Test columns:", X_train_prep.columns.equals(X_test_prep.columns))
print("Columns in train but not in test:", set(X_train_prep.columns) - set(X_test_prep.columns))
print("Columns in test but not in train:", set(X_test_prep.columns) - set(X_train_prep.columns))

Train columns == Test columns: True
Columns in train but not in test: set()
Columns in test but not in train: set()


In [6]:
# Read
y_train.to_csv(c.RICKD_MODEL_1_Y_TRAIN_FILE_3)
y_test.to_csv(c.RICKD_MODEL_1_Y_TEST_FILE_3)

In [7]:
# Save
X_train_prep.to_csv(c.RICKD_MODEL_1_X_TRAIN_PREPROCESSED_FILE_3)
X_test_prep.to_csv(c.RICKD_MODEL_1_X_TEST_PREPROCESSED_FILE_3)

In [8]:
# Read
X_train_prep_df = pd.read_csv(c.RICKD_MODEL_1_X_TRAIN_PREPROCESSED_FILE_3, index_col=0)
X_test_prep_df = pd.read_csv(c.RICKD_MODEL_1_X_TEST_PREPROCESSED_FILE_3, index_col=0)